In [27]:
import numpy as np
import pandas as pd
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================
ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

LAG_PREFIXES = ["lag5", "lag10", "lag21", "lag63"]
TARGET_PREFIX = "target"

INPUT_FILES = {
    10: "../proxy/realized_cov_target_h10_lag_5_10_21_63.csv",
    21: "../proxy/realized_cov_target_h21_lag_5_10_21_63.csv",
    63: "../proxy/realized_cov_target_h63_lag_5_10_21_63.csv",
}

OUTPUT_TEMPLATE = "../proxy/realized_chol_target_h{h}_lag_5_10_21_63.csv"

Grid search for SVR 

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =========================================================
# GRID SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average QLIKE, keep Frobenius too
# Validation period: 2019-2020 in current settings
# Compare against realized covariance proxy
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 10

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2020-12-31"

WINDOW_GRID = [252]
C_GRID = [0.1, 1, 10]
GAMMA_GRID = ["scale", 0.1]
EPSILON_GRID = [0.001]

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = pd.read_csv(CHOL_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
proxy = pd.read_csv(PROXY_PATH, parse_dates=["Date"]).sort_values("Date").set_index("Date")

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

def make_spd(M, eps=RIDGE_EPS):
    M = 0.5 * (np.asarray(M, dtype=float) + np.asarray(M, dtype=float).T)
    eigvals = np.linalg.eigvalsh(M)
    min_eig = eigvals.min()
    if min_eig <= eps:
        M = M + np.eye(M.shape[0]) * (eps - min_eig + eps)
    return M

def qlike_loss(S_true, H_pred, eps=RIDGE_EPS):
    """
    Matrix QLIKE:
        tr(S_true @ H_pred^{-1}) - logdet(S_true @ H_pred^{-1}) - n

    Computed in a numerically stable way as:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, eps=eps)
    H_pred = make_spd(H_pred, eps=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    qlike = trace_term - logdet_s + logdet_h - n
    return float(qlike)

lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):

    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0

    for p in validation_dates:

        p_loc = chol.index[chol["Date"] == p][0]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:

            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:

                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )

                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]

        L = np.zeros((n_assets, n_assets))

        for a, b in lower_pairs:

            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs = scaler_x.transform(X)
            y_scaled = model.predict(Xs)

            y_pred = scaler_y.inverse_transform(
                y_scaled.reshape(-1, 1)
            )[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma, eps=RIDGE_EPS)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)
        S_true = make_spd(S_true, eps=RIDGE_EPS)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma, eps=RIDGE_EPS)

        if np.isfinite(frob) and np.isfinite(qlike):
            frob_list.append(frob)
            qlike_list.append(qlike)
            n_forecasts += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits
    }

# ---------------------------------------------------------
# BUILD GRID
# ---------------------------------------------------------
specs = list(product(WINDOW_GRID, C_GRID, GAMMA_GRID, EPSILON_GRID))

print("Total SVR specs:", len(specs))
print()

# ---------------------------------------------------------
# GRID SEARCH
# ---------------------------------------------------------
results = []

for window, C, gamma, epsilon in tqdm(specs, desc=f"SVR grid search (h={TARGET_H})"):
    res = evaluate_svr_spec(window, C, gamma, epsilon)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# RANK MODELS
# ---------------------------------------------------------
results_df = results_df.sort_values(
    ["avg_qlike", "avg_frobenius", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

print("\n=========================================================")
print("SVR MODELS RANKED BY QLIKE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print("=========================================================\n")

print(results_df.to_string(index=False))

best = results_df.iloc[0]

print("\n=========================================================")
print("BEST SVR MODEL")
print("=========================================================")
print(best.to_string())

Loaded Cholesky data: ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h10.csv
Validation: 2019-01-02 00:00:00 -> 2020-12-31 00:00:00

Total SVR specs: 42



SVR grid search (h=10):   0%|          | 0/42 [00:00<?, ?it/s]


SVR MODELS RANKED BY QLIKE
Forecast horizon: 10
Validation period: 2019-01-01 -> 2020-12-31

 window    C gamma  epsilon  avg_qlike  avg_frobenius  n_forecasts  n_refits
    252  0.1   0.1    0.001  42.619132       0.002438          505       505
    315  0.1   0.1    0.001  42.749587       0.002450          505       505
    189  0.1   0.1    0.001  43.391031       0.002430          505       505
    315  0.1 scale    0.001  43.504028       0.002456          505       505
    378  0.1   0.1    0.001  44.020043       0.002465          499       499
    252  0.1 scale    0.001  44.970398       0.002443          505       505
    189  0.1 scale    0.001  45.162441       0.002427          505       505
    378  0.1 scale    0.001  45.458761       0.002459          499       499
    441  0.1   0.1    0.001  46.216302       0.002705          436       436
    441  0.1 scale    0.001  48.863427       0.002695          436       436
    504  0.1   0.1    0.001  49.244300       0.002575      

Getting the cholesky csv, no need to run, have it stored

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

LAG_PREFIXES = ["lag5", "lag10", "lag21", "lag63"]
TARGET_PREFIX = "target"

INPUT_FILES = {
    10: "../proxy/realized_cov_target_h10_lag_5_10_21_63.csv",
    21: "../proxy/realized_cov_target_h21_lag_5_10_21_63.csv",
    63: "../proxy/realized_cov_target_h63_lag_5_10_21_63.csv",
}

OUTPUT_TEMPLATE = "../proxy/realized_chol_target_h{h}_lag_5_10_21_63.csv"

# Small ridge for numerical stability before Cholesky
BASE_RIDGE = 1e-12
MAX_RIDGE_TRIES = 8


# =========================================================
# HELPERS
# =========================================================
def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_cov_cols(prefix, asset_order):
    return [f"{prefix}_cov_{a}__{b}" for a, b in build_full_pairs(asset_order)]


def build_chol_cols(prefix, asset_order):
    return [f"{prefix}_chol_{a}__{b}" for a, b in build_lower_tri_pairs(asset_order)]


def extract_cov_matrix_from_row(row, prefix, asset_order):
    n = len(asset_order)
    mat = np.zeros((n, n), dtype=float)

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            col = f"{prefix}_cov_{a}__{b}"
            mat[i, j] = row[col]

    # Force exact symmetry in case of tiny numerical mismatch
    mat = 0.5 * (mat + mat.T)
    return mat


def safe_cholesky(mat, base_ridge=1e-12, max_tries=8):
    """
    Try Cholesky with increasing ridge if needed.
    Returns lower-triangular matrix L.
    """
    n = mat.shape[0]
    eye = np.eye(n)
    ridge = base_ridge

    for _ in range(max_tries):
        try:
            return np.linalg.cholesky(mat + ridge * eye)
        except np.linalg.LinAlgError:
            ridge *= 10

    min_eig = np.min(np.linalg.eigvalsh(mat))
    raise np.linalg.LinAlgError(
        f"Cholesky failed even after ridge escalation. "
        f"Minimum eigenvalue before ridge: {min_eig:.6e}"
    )


def flatten_cholesky_lower_tri(L, asset_order):
    out = {}

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                out[f"{a}__{b}"] = float(L[i, j])

    return out


def get_metadata_columns(df):
    preferred = [
        "Date",
        "lag5_start", "lag5_end",
        "lag10_start", "lag10_end",
        "lag21_start", "lag21_end",
        "lag63_start", "lag63_end",
        "target_start", "target_end",
    ]
    return [c for c in preferred if c in df.columns]


def convert_cov_df_to_chol_df(df_cov, asset_order,
                              base_ridge=1e-12, max_ridge_tries=8):
    metadata_cols = get_metadata_columns(df_cov)
    prefixes = [TARGET_PREFIX] + LAG_PREFIXES

    expected_cov_cols = []
    for prefix in prefixes:
        expected_cov_cols.extend(build_cov_cols(prefix, asset_order))

    missing = [c for c in expected_cov_cols if c not in df_cov.columns]
    if missing:
        raise ValueError(f"Missing expected covariance columns, e.g. {missing[:10]}")

    output_rows = []

    for row_idx, (_, row) in enumerate(df_cov.iterrows()):
        out_row = {}

        # Keep metadata
        for c in metadata_cols:
            out_row[c] = row[c]

        # Convert each covariance block to lower-triangular Cholesky entries
        for prefix in prefixes:
            cov_mat = extract_cov_matrix_from_row(row, prefix, asset_order)
            L = safe_cholesky(
                cov_mat,
                base_ridge=base_ridge,
                max_tries=max_ridge_tries
            )

            chol_entries = flatten_cholesky_lower_tri(L, asset_order)

            for pair_name, value in chol_entries.items():
                out_row[f"{prefix}_chol_{pair_name}"] = value

        output_rows.append(out_row)

        if (row_idx + 1) % 250 == 0:
            print(f"Processed {row_idx + 1} rows...")

    df_chol = pd.DataFrame(output_rows)

    # Final column order
    final_cols = metadata_cols.copy()
    for prefix in prefixes:
        final_cols.extend(build_chol_cols(prefix, asset_order))

    df_chol = df_chol[final_cols].copy()
    return df_chol


def convert_one_horizon(h, input_path, output_path, asset_order,
                        base_ridge=1e-12, max_ridge_tries=8):
    print("=" * 70)
    print(f"Converting horizon h={h}")
    print("Input :", input_path)
    print("Output:", output_path)

    df_cov = pd.read_csv(input_path, parse_dates=["Date"])
    print("Loaded shape:", df_cov.shape)

    df_chol = convert_cov_df_to_chol_df(
        df_cov=df_cov,
        asset_order=asset_order,
        base_ridge=base_ridge,
        max_ridge_tries=max_ridge_tries
    )

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df_chol.to_csv(output_path, index=False)

    print("Saved shape:", df_chol.shape)
    print("Saved to   :", output_path)
    print()


# =========================================================
# RUN FOR ALL HORIZONS
# =========================================================
for h, input_path in INPUT_FILES.items():
    output_path = OUTPUT_TEMPLATE.format(h=h)

    convert_one_horizon(
        h=h,
        input_path=input_path,
        output_path=output_path,
        asset_order=ASSET_ORDER,
        base_ridge=BASE_RIDGE,
        max_ridge_tries=MAX_RIDGE_TRIES
    )

Converting horizon h=10
Input : ../proxy/realized_cov_target_h10_lag_5_10_21_63.csv
Output: ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv
Loaded shape: (2132, 331)
Processed 250 rows...
Processed 500 rows...
Processed 750 rows...
Processed 1000 rows...
Processed 1250 rows...
Processed 1500 rows...
Processed 1750 rows...
Processed 2000 rows...
Saved shape: (2132, 191)
Saved to   : ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv

Converting horizon h=21
Input : ../proxy/realized_cov_target_h21_lag_5_10_21_63.csv
Output: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded shape: (2121, 331)
Processed 250 rows...
Processed 500 rows...
Processed 750 rows...
Processed 1000 rows...
Processed 1250 rows...
Processed 1500 rows...
Processed 1750 rows...
Processed 2000 rows...
Saved shape: (2121, 191)
Saved to   : ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv

Converting horizon h=63
Input : ../proxy/realized_cov_target_h63_lag_5_10_21_63.csv
Output: ../proxy/realized_ch

Grid search using Optuna 

In [7]:
import numpy as np
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# =========================================================
# OPTUNA SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average QLIKE, keep Frobenius too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# SPD is enforced on forecast only if needed
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 10

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12
RANDOM_STATE = 42
N_TRIALS = 50

SAVE_TRIALS_PATH = f"../results/svr_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/svr_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M


def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))


def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)

    # Only repair if needed
    try:
        np.linalg.cholesky(matrix)
        return matrix
    except np.linalg.LinAlgError:
        pass

    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, min_eig)
    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd += np.eye(spd.shape[0]) * jitter
    spd = 0.5 * (spd + spd.T)
    return spd


def qlike_loss(S_true, H_pred):
    """
    Assumes H_pred has already been projected to SPD with make_spd().
    Proxy/S_true is used as-is.
    """
    n = S_true.shape[0]

    S_true = 0.5 * (np.asarray(S_true, dtype=float) + np.asarray(S_true, dtype=float).T)
    H_pred = 0.5 * (np.asarray(H_pred, dtype=float) + np.asarray(H_pred, dtype=float).T)

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)


lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

date_to_loc = {d: i for i, d in enumerate(chol["Date"])}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0
    n_invalid_qlike = 0

    for p in validation_dates:
        p_loc = date_to_loc[p]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:
            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:
                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )
                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets), dtype=float)

        for a, b in lower_pairs:
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)
            y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma)

        if np.isfinite(frob):
            frob_list.append(frob)
            n_forecasts += 1

        if np.isfinite(qlike):
            qlike_list.append(qlike)
        else:
            n_invalid_qlike += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits,
        "n_valid_qlike": len(qlike_list),
        "n_invalid_qlike": n_invalid_qlike
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)

    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "numeric"])
    if gamma_mode == "scale":
        gamma = "scale"
    else:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)

    epsilon = trial.suggest_float("epsilon", 1e-5, 1e-1, log=True)

    res = evaluate_svr_spec(
        window=window,
        C=C,
        gamma=gamma,
        epsilon=epsilon
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])
    trial.set_user_attr("n_valid_qlike", res["n_valid_qlike"])
    trial.set_user_attr("n_invalid_qlike", res["n_invalid_qlike"])
    trial.set_user_attr("gamma_used", gamma)

    trial_results.append(res)

    if not np.isfinite(res["avg_qlike"]):
        return 1e12

    return res["avg_qlike"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]
trials_df["n_valid_qlike"] = [t.user_attrs.get("n_valid_qlike", np.nan) for t in study.trials]
trials_df["n_invalid_qlike"] = [t.user_attrs.get("n_invalid_qlike", np.nan) for t in study.trials]
trials_df["gamma_used"] = [t.user_attrs.get("gamma_used", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "avg_qlike"})

rename_map = {
    "params_window": "window",
    "params_C": "C",
    "params_gamma_mode": "gamma_mode",
    "params_gamma": "gamma",
    "params_epsilon": "epsilon",
}
trials_df = trials_df.rename(columns=rename_map)

keep_cols = [
    "number",
    "state",
    "avg_qlike",
    "avg_frobenius",
    "window",
    "C",
    "gamma_mode",
    "gamma",
    "gamma_used",
    "epsilon",
    "n_forecasts",
    "n_refits",
    "n_valid_qlike",
    "n_invalid_qlike",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

trials_df = trials_df.sort_values(
    ["avg_qlike", "avg_frobenius", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------
trials_df.to_csv(SAVE_TRIALS_PATH, index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_qlike"] = study.best_value
best_params_df["best_avg_frobenius"] = (
    trials_df.iloc[0]["avg_frobenius"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_forecasts"] = (
    trials_df.iloc[0]["n_forecasts"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_refits"] = (
    trials_df.iloc[0]["n_refits"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_valid_qlike"] = (
    trials_df.iloc[0]["n_valid_qlike"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_invalid_qlike"] = (
    trials_df.iloc[0]["n_invalid_qlike"] if len(trials_df) > 0 else np.nan
)

best_gamma_mode = study.best_params.get("gamma_mode")
if best_gamma_mode == "scale":
    best_gamma_used = "scale"
else:
    best_gamma_used = study.best_params.get("gamma")

best_params_df["gamma_used"] = best_gamma_used
best_params_df.to_csv(SAVE_BEST_PATH, index=False)

print("\n=========================================================")
print("OPTUNA SVR SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by QLIKE:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg QLIKE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  gamma_used:", best_gamma_used)

print("\nSaved trial results to:", SAVE_TRIALS_PATH)
print("Saved best parameters to:", SAVE_BEST_PATH)

[I 2026-04-14 14:45:36,961] A new study created in memory with name: no-name-89f0b9f0-0e43-41a1-a6b2-eba1e6e1c356


Loaded Cholesky data: ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h10.csv
Validation: 2019-01-02 00:00:00 -> 2020-12-17 00:00:00
Refit every: 1
Number of Optuna trials: 50



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-14 14:46:30,255] Trial 0 finished with value: 68.40815841015836 and parameters: {'window': 189, 'C': 157.41890047456639, 'gamma_mode': 'numeric', 'gamma': 0.00012674255898937226, 'epsilon': 0.07579479953347998}. Best is trial 0 with value: 68.40815841015836.
[I 2026-04-14 14:47:05,831] Trial 1 finished with value: 52.04251688842246 and parameters: {'window': 126, 'C': 0.055895242052179224, 'gamma_mode': 'scale', 'epsilon': 0.00014742753159914678}. Best is trial 1 with value: 52.04251688842246.
[I 2026-04-14 14:48:47,341] Trial 2 finished with value: 91.3058423304032 and parameters: {'window': 252, 'C': 4.418441521199721, 'gamma_mode': 'scale', 'epsilon': 0.062451395747430666}. Best is trial 1 with value: 52.04251688842246.
[I 2026-04-14 14:49:25,840] Trial 3 finished with value: 65.91102363089334 and parameters: {'window': 126, 'C': 0.9355380606452183, 'gamma_mode': 'numeric', 'gamma': 0.0019674328025306126, 'epsilon': 0.004467752817973905}. Best is trial 1 with value: 52.04

In [8]:
import numpy as np
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# =========================================================
# OPTUNA SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average QLIKE, keep Frobenius too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# SPD is enforced on forecast only if needed
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 21

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12
RANDOM_STATE = 42
N_TRIALS = 50

SAVE_TRIALS_PATH = f"../results/svr_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/svr_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M


def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))


def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)

    # Only repair if needed
    try:
        np.linalg.cholesky(matrix)
        return matrix
    except np.linalg.LinAlgError:
        pass

    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, min_eig)
    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd += np.eye(spd.shape[0]) * jitter
    spd = 0.5 * (spd + spd.T)
    return spd


def qlike_loss(S_true, H_pred):
    """
    Assumes H_pred has already been projected to SPD with make_spd().
    Proxy/S_true is used as-is.
    """
    n = S_true.shape[0]

    S_true = 0.5 * (np.asarray(S_true, dtype=float) + np.asarray(S_true, dtype=float).T)
    H_pred = 0.5 * (np.asarray(H_pred, dtype=float) + np.asarray(H_pred, dtype=float).T)

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)


lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

date_to_loc = {d: i for i, d in enumerate(chol["Date"])}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0
    n_invalid_qlike = 0

    for p in validation_dates:
        p_loc = date_to_loc[p]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:
            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:
                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )
                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets), dtype=float)

        for a, b in lower_pairs:
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)
            y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma)

        if np.isfinite(frob):
            frob_list.append(frob)
            n_forecasts += 1

        if np.isfinite(qlike):
            qlike_list.append(qlike)
        else:
            n_invalid_qlike += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits,
        "n_valid_qlike": len(qlike_list),
        "n_invalid_qlike": n_invalid_qlike
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)

    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "numeric"])
    if gamma_mode == "scale":
        gamma = "scale"
    else:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)

    epsilon = trial.suggest_float("epsilon", 1e-5, 1e-1, log=True)

    res = evaluate_svr_spec(
        window=window,
        C=C,
        gamma=gamma,
        epsilon=epsilon
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])
    trial.set_user_attr("n_valid_qlike", res["n_valid_qlike"])
    trial.set_user_attr("n_invalid_qlike", res["n_invalid_qlike"])
    trial.set_user_attr("gamma_used", gamma)

    trial_results.append(res)

    if not np.isfinite(res["avg_qlike"]):
        return 1e12

    return res["avg_qlike"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]
trials_df["n_valid_qlike"] = [t.user_attrs.get("n_valid_qlike", np.nan) for t in study.trials]
trials_df["n_invalid_qlike"] = [t.user_attrs.get("n_invalid_qlike", np.nan) for t in study.trials]
trials_df["gamma_used"] = [t.user_attrs.get("gamma_used", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "avg_qlike"})

rename_map = {
    "params_window": "window",
    "params_C": "C",
    "params_gamma_mode": "gamma_mode",
    "params_gamma": "gamma",
    "params_epsilon": "epsilon",
}
trials_df = trials_df.rename(columns=rename_map)

keep_cols = [
    "number",
    "state",
    "avg_qlike",
    "avg_frobenius",
    "window",
    "C",
    "gamma_mode",
    "gamma",
    "gamma_used",
    "epsilon",
    "n_forecasts",
    "n_refits",
    "n_valid_qlike",
    "n_invalid_qlike",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

trials_df = trials_df.sort_values(
    ["avg_qlike", "avg_frobenius", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------
trials_df.to_csv(SAVE_TRIALS_PATH, index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_qlike"] = study.best_value
best_params_df["best_avg_frobenius"] = (
    trials_df.iloc[0]["avg_frobenius"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_forecasts"] = (
    trials_df.iloc[0]["n_forecasts"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_refits"] = (
    trials_df.iloc[0]["n_refits"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_valid_qlike"] = (
    trials_df.iloc[0]["n_valid_qlike"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_invalid_qlike"] = (
    trials_df.iloc[0]["n_invalid_qlike"] if len(trials_df) > 0 else np.nan
)

best_gamma_mode = study.best_params.get("gamma_mode")
if best_gamma_mode == "scale":
    best_gamma_used = "scale"
else:
    best_gamma_used = study.best_params.get("gamma")

best_params_df["gamma_used"] = best_gamma_used
best_params_df.to_csv(SAVE_BEST_PATH, index=False)

print("\n=========================================================")
print("OPTUNA SVR SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by QLIKE:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg QLIKE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  gamma_used:", best_gamma_used)

print("\nSaved trial results to:", SAVE_TRIALS_PATH)
print("Saved best parameters to:", SAVE_BEST_PATH)

[I 2026-04-14 16:42:56,072] A new study created in memory with name: no-name-47df9d65-a5cc-41a4-b7a6-72205cd2e1cc


Loaded Cholesky data: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h21.csv
Validation: 2019-01-02 00:00:00 -> 2020-12-02 00:00:00
Refit every: 1
Number of Optuna trials: 50



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-14 16:43:44,735] Trial 0 finished with value: 38402.91361007972 and parameters: {'window': 189, 'C': 157.41890047456639, 'gamma_mode': 'numeric', 'gamma': 0.00012674255898937226, 'epsilon': 0.07579479953347998}. Best is trial 0 with value: 38402.91361007972.
[I 2026-04-14 16:44:18,786] Trial 1 finished with value: 26.861800631263996 and parameters: {'window': 126, 'C': 0.055895242052179224, 'gamma_mode': 'scale', 'epsilon': 0.00014742753159914678}. Best is trial 1 with value: 26.861800631263996.
[I 2026-04-14 16:45:47,530] Trial 2 finished with value: 114.65814744052457 and parameters: {'window': 252, 'C': 4.418441521199721, 'gamma_mode': 'scale', 'epsilon': 0.062451395747430666}. Best is trial 1 with value: 26.861800631263996.
[I 2026-04-14 16:46:22,056] Trial 3 finished with value: 31.59203817570077 and parameters: {'window': 126, 'C': 0.9355380606452183, 'gamma_mode': 'numeric', 'gamma': 0.0019674328025306126, 'epsilon': 0.004467752817973905}. Best is trial 1 with value: 

In [9]:
import numpy as np
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# =========================================================
# OPTUNA SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average QLIKE, keep Frobenius too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# SPD is enforced on forecast only if needed
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 63

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12
RANDOM_STATE = 42
N_TRIALS = 50

SAVE_TRIALS_PATH = f"../results/svr_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/svr_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M


def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))


def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)

    # Only repair if needed
    try:
        np.linalg.cholesky(matrix)
        return matrix
    except np.linalg.LinAlgError:
        pass

    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, min_eig)
    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd += np.eye(spd.shape[0]) * jitter
    spd = 0.5 * (spd + spd.T)
    return spd


def qlike_loss(S_true, H_pred):
    """
    Assumes H_pred has already been projected to SPD with make_spd().
    Proxy/S_true is used as-is.
    """
    n = S_true.shape[0]

    S_true = 0.5 * (np.asarray(S_true, dtype=float) + np.asarray(S_true, dtype=float).T)
    H_pred = 0.5 * (np.asarray(H_pred, dtype=float) + np.asarray(H_pred, dtype=float).T)

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)


lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

date_to_loc = {d: i for i, d in enumerate(chol["Date"])}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0
    n_invalid_qlike = 0

    for p in validation_dates:
        p_loc = date_to_loc[p]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:
            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:
                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )
                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets), dtype=float)

        for a, b in lower_pairs:
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)
            y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma)

        if np.isfinite(frob):
            frob_list.append(frob)
            n_forecasts += 1

        if np.isfinite(qlike):
            qlike_list.append(qlike)
        else:
            n_invalid_qlike += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits,
        "n_valid_qlike": len(qlike_list),
        "n_invalid_qlike": n_invalid_qlike
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)

    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "numeric"])
    if gamma_mode == "scale":
        gamma = "scale"
    else:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)

    epsilon = trial.suggest_float("epsilon", 1e-5, 1e-1, log=True)

    res = evaluate_svr_spec(
        window=window,
        C=C,
        gamma=gamma,
        epsilon=epsilon
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])
    trial.set_user_attr("n_valid_qlike", res["n_valid_qlike"])
    trial.set_user_attr("n_invalid_qlike", res["n_invalid_qlike"])
    trial.set_user_attr("gamma_used", gamma)

    trial_results.append(res)

    if not np.isfinite(res["avg_qlike"]):
        return 1e12

    return res["avg_qlike"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]
trials_df["n_valid_qlike"] = [t.user_attrs.get("n_valid_qlike", np.nan) for t in study.trials]
trials_df["n_invalid_qlike"] = [t.user_attrs.get("n_invalid_qlike", np.nan) for t in study.trials]
trials_df["gamma_used"] = [t.user_attrs.get("gamma_used", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "avg_qlike"})

rename_map = {
    "params_window": "window",
    "params_C": "C",
    "params_gamma_mode": "gamma_mode",
    "params_gamma": "gamma",
    "params_epsilon": "epsilon",
}
trials_df = trials_df.rename(columns=rename_map)

keep_cols = [
    "number",
    "state",
    "avg_qlike",
    "avg_frobenius",
    "window",
    "C",
    "gamma_mode",
    "gamma",
    "gamma_used",
    "epsilon",
    "n_forecasts",
    "n_refits",
    "n_valid_qlike",
    "n_invalid_qlike",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

trials_df = trials_df.sort_values(
    ["avg_qlike", "avg_frobenius", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------
trials_df.to_csv(SAVE_TRIALS_PATH, index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_qlike"] = study.best_value
best_params_df["best_avg_frobenius"] = (
    trials_df.iloc[0]["avg_frobenius"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_forecasts"] = (
    trials_df.iloc[0]["n_forecasts"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_refits"] = (
    trials_df.iloc[0]["n_refits"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_valid_qlike"] = (
    trials_df.iloc[0]["n_valid_qlike"] if len(trials_df) > 0 else np.nan
)
best_params_df["n_invalid_qlike"] = (
    trials_df.iloc[0]["n_invalid_qlike"] if len(trials_df) > 0 else np.nan
)

best_gamma_mode = study.best_params.get("gamma_mode")
if best_gamma_mode == "scale":
    best_gamma_used = "scale"
else:
    best_gamma_used = study.best_params.get("gamma")

best_params_df["gamma_used"] = best_gamma_used
best_params_df.to_csv(SAVE_BEST_PATH, index=False)

print("\n=========================================================")
print("OPTUNA SVR SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by QLIKE:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg QLIKE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  gamma_used:", best_gamma_used)

print("\nSaved trial results to:", SAVE_TRIALS_PATH)
print("Saved best parameters to:", SAVE_BEST_PATH)

[I 2026-04-14 19:20:13,059] A new study created in memory with name: no-name-5bf66e1f-512e-43a9-865c-00fe1e4d9b65


Loaded Cholesky data: ../proxy/realized_chol_target_h63_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h63.csv
Validation: 2019-01-02 00:00:00 -> 2020-10-02 00:00:00
Refit every: 1
Number of Optuna trials: 50



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-14 19:20:59,872] Trial 0 finished with value: 1016136296206.0481 and parameters: {'window': 189, 'C': 157.41890047456639, 'gamma_mode': 'numeric', 'gamma': 0.00012674255898937226, 'epsilon': 0.07579479953347998}. Best is trial 0 with value: 1016136296206.0481.
[I 2026-04-14 19:21:32,636] Trial 1 finished with value: 20.393579802159923 and parameters: {'window': 126, 'C': 0.055895242052179224, 'gamma_mode': 'scale', 'epsilon': 0.00014742753159914678}. Best is trial 1 with value: 20.393579802159923.
[I 2026-04-14 19:22:55,647] Trial 2 finished with value: 4629.598854949768 and parameters: {'window': 252, 'C': 4.418441521199721, 'gamma_mode': 'scale', 'epsilon': 0.062451395747430666}. Best is trial 1 with value: 20.393579802159923.
[I 2026-04-14 19:23:27,611] Trial 3 finished with value: 3710331.7504561353 and parameters: {'window': 126, 'C': 0.9355380606452183, 'gamma_mode': 'numeric', 'gamma': 0.0019674328025306126, 'epsilon': 0.004467752817973905}. Best is trial 1 with value

Running forecasting scheme for SVR, insert best parameters manually

In [16]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================
TARGET_H = 63

ASSET_ORDER = [
    "US_EQUITY","INTL_EQUITY","US_BONDS","INTL_BONDS",
    "US_REIT","INTL_REIT","GOLD","BTC"
]

LAGS = (10,21,63)

TEST_START = "2021-01-01"

TRAINING_WINDOW = 315
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12

KERNEL = "rbf"


CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
SAVE_PATH = f"../results/svr_cov_forecast_h{TARGET_H}.csv"
C       = 0.00403
GAMMA   = "scale"
EPSILON = 0.0000177

# =========================================================
# HELPERS
# =========================================================
def build_lower_pairs(asset_order):

    pairs = []
    for i,a in enumerate(asset_order):
        for j,b in enumerate(asset_order):
            if i >= j:
                pairs.append((a,b))
    return pairs


def build_full_pairs(asset_order):

    return [(a,b) for a in asset_order for b in asset_order]


def reconstruct_covariance(L):

    return L @ L.T

def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)
    
    # Kjør bare eigendekomposisjon hvis nødvendig
    try:
        np.linalg.cholesky(matrix)
        return matrix
    except np.linalg.LinAlgError:
        pass
    
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, min_eig)
    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd += np.eye(spd.shape[0]) * jitter
    spd = 0.5 * (spd + spd.T)
    return spd


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(CHOL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

lower_pairs = build_lower_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

test_start = pd.Timestamp(TEST_START)

candidate_dates = df.loc[df["Date"] >= test_start,"Date"]


# =========================================================
# ROLLING FORECAST
# =========================================================
forecast_rows = []

models = None
scalers_x = None
scalers_y = None
last_refit_index = None


for p in candidate_dates:

    p_idx = df.index[df["Date"] == p][0]

    should_refit = (
        models is None
        or last_refit_index is None
        or (p_idx - last_refit_index) >= REFIT_EVERY
    )

    if should_refit:

        train_end = p_idx - TARGET_H
        train_start = train_end - TRAINING_WINDOW + 1

        if train_start < 0:
            continue

        train = df.iloc[train_start:train_end+1]

        models = {}
        scalers_x = {}
        scalers_y = {}

        for a,b in lower_pairs:

            target_col = f"target_chol_{a}__{b}"
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            X = train[lag_cols].values
            y = train[target_col].values.reshape(-1,1)

            scaler_x = StandardScaler()
            scaler_y = StandardScaler()

            Xs = scaler_x.fit_transform(X)
            ys = scaler_y.fit_transform(y).ravel()

            model = SVR(
                kernel=KERNEL,
                C=C,
                epsilon=EPSILON,
                gamma=GAMMA
            )

            model.fit(Xs,ys)

            models[(a,b)] = model
            scalers_x[(a,b)] = scaler_x
            scalers_y[(a,b)] = scaler_y

        last_refit_index = p_idx

        print(
            f"Refit at {p.date()} | "
            f"train rows {train.index.min()} -> {train.index.max()}"
        )


    # =====================================================
    # PREDICT CHOLESKY FACTOR
    # =====================================================
    n = len(ASSET_ORDER)
    L_pred = np.zeros((n,n))

    row = df.iloc[p_idx]

    for a,b in lower_pairs:

        lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
        X_pred = row[lag_cols].values.reshape(1,-1)

        scaler_x = scalers_x[(a,b)]
        scaler_y = scalers_y[(a,b)]
        model = models[(a,b)]

        Xs = scaler_x.transform(X_pred)

        y_scaled = model.predict(Xs)

        y_pred = scaler_y.inverse_transform(
            y_scaled.reshape(-1,1)
        )[0,0]

        i = ASSET_ORDER.index(a)
        j = ASSET_ORDER.index(b)

        L_pred[i,j] = y_pred


    # =====================================================
    # RECONSTRUCT COVARIANCE
    # =====================================================
    Sigma = reconstruct_covariance(L_pred)
    Sigma = make_spd(Sigma)

    out_row = {"Date":p}

    for i,a in enumerate(ASSET_ORDER):
        for j,b in enumerate(ASSET_ORDER):

            out_row[f"cov_{a}__{b}"] = Sigma[i,j]

    forecast_rows.append(out_row)


# =========================================================
# SAVE FORECASTS
# =========================================================
forecast_df = pd.DataFrame(forecast_rows)

cols = ["Date"] + [
    f"cov_{a}__{b}" for a,b in full_pairs
]

forecast_df = forecast_df[cols]

Path("../results").mkdir(exist_ok=True)

forecast_df.to_csv(SAVE_PATH,index=False)

print()
print("Saved forecasts to:",SAVE_PATH)
print("Shape:",forecast_df.shape)

Refit at 2021-01-04 | train rows 509 -> 823
Refit at 2021-01-05 | train rows 510 -> 824
Refit at 2021-01-06 | train rows 511 -> 825
Refit at 2021-01-07 | train rows 512 -> 826
Refit at 2021-01-08 | train rows 513 -> 827
Refit at 2021-01-11 | train rows 514 -> 828
Refit at 2021-01-12 | train rows 515 -> 829
Refit at 2021-01-13 | train rows 516 -> 830
Refit at 2021-01-14 | train rows 517 -> 831
Refit at 2021-01-15 | train rows 518 -> 832
Refit at 2021-01-19 | train rows 519 -> 833
Refit at 2021-01-20 | train rows 520 -> 834
Refit at 2021-01-21 | train rows 521 -> 835
Refit at 2021-01-22 | train rows 522 -> 836
Refit at 2021-01-25 | train rows 523 -> 837
Refit at 2021-01-26 | train rows 524 -> 838
Refit at 2021-01-27 | train rows 525 -> 839
Refit at 2021-01-28 | train rows 526 -> 840
Refit at 2021-01-29 | train rows 527 -> 841
Refit at 2021-02-01 | train rows 528 -> 842
Refit at 2021-02-02 | train rows 529 -> 843
Refit at 2021-02-03 | train rows 530 -> 844
Refit at 2021-02-04 | train rows